# 01 — Text-to-SQL system architecture

| | |
|---|---|
| **Track** | Applied Labs |
| **Domain** | AG — AI Data Analyst |
| **Level** | Advanced |
| **Time** | ~35 min |
| **Prerequisites** | [00_first_principles.ipynb](./00_first_principles.ipynb) |
| **Open in Colab** | [![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai2026/castalia/blob/main/labs/06_ai_data_analyst/01_architecture.ipynb) |

**What you'll learn:**
- Full system architecture for an AI data analyst
- Schema indexing with retrieval-augmented generation
- Question disambiguation strategies
- SQL generation with schema context and few-shot prompting
- Execution, validation, and self-correction agent design

## 1 · Setup

In [ ]:
!pip install -q "openai>=1.0.0" "pandas>=2.0.0"

import os
import json
import sqlite3
import pandas as pd
from typing import Optional
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"

print(f"✅ Setup complete — model: {MODEL}")

## 2 · System architecture

```
┌─────────────┐     ┌──────────────────┐     ┌───────────────┐
│   User       │────▶│  Question        │────▶│  Schema       │
│   Question   │     │  Disambiguator   │     │  Retriever    │
└─────────────┘     └──────────────────┘     └───────┬───────┘
                                                      │
                    ┌──────────────────┐              │
                    │  SQL Generator   │◀─────────────┘
                    │  (LLM + prompt)  │
                    └────────┬─────────┘
                             │
                    ┌────────▼─────────┐
                    │  SQL Executor    │
                    │  (SQLite/PG)     │
                    └────────┬─────────┘
                             │
                    ┌────────▼─────────┐     ┌───────────────┐
                    │  Validator       │────▶│  Self-         │
                    │  (type, empty,   │     │  Correction    │──┐
                    │   row count)     │     │  Agent         │  │
                    └────────┬─────────┘     └───────────────┘  │
                             │ pass                     ▲        │
                    ┌────────▼─────────┐                │ retry  │
                    │  Narrator        │                └────────┘
                    │  (LLM insights)  │
                    └────────┬─────────┘
                             │
                    ┌────────▼─────────┐
                    │  Chart Generator │
                    │  (Matplotlib)    │
                    └─────────────────┘
```

**Six components**, each with a clear contract. We'll design each one.

In [ ]:
# Define the component interfaces as dataclasses for clarity

from dataclasses import dataclass, field


@dataclass
class SchemaContext:
    """Retrieved schema information relevant to a question."""
    tables: list[str]
    columns: dict[str, list[str]]  # table → columns
    relationships: list[str]
    glossary_terms: list[str]
    example_queries: list[str]


@dataclass
class DisambiguatedQuestion:
    """A question with explicit assumptions stated."""
    original: str
    clarified: str
    assumptions: list[str]


@dataclass
class SQLResult:
    """Result of SQL generation and execution."""
    sql: str
    success: bool
    data: Optional[pd.DataFrame] = None
    error: Optional[str] = None
    attempts: int = 1


@dataclass
class AnalystResponse:
    """Complete response from the AI data analyst."""
    question: str
    disambiguation: DisambiguatedQuestion
    sql: str
    data: Optional[pd.DataFrame] = None
    narration: str = ""
    chart_type: str = "table"
    attempts: int = 1


print("✅ Component interfaces defined")
for cls in [SchemaContext, DisambiguatedQuestion, SQLResult, AnalystResponse]:
    print(f"  {cls.__name__}: {list(cls.__dataclass_fields__.keys())}")

## 3 · Schema indexing with RAG

The schema retriever finds **relevant tables, columns, and business terms**
for a given question. This focused context is critical — sending the entire
schema wastes tokens and confuses the model on large databases.

**What we index:**
- Table descriptions and purposes
- Column names, types, and semantic descriptions
- Foreign-key relationships and join paths
- Business glossary terms and their SQL definitions
- Example question → SQL pairs for few-shot learning

In [ ]:
# Build a schema documentation store

SCHEMA_STORE: dict[str, dict] = {
    "tables": {
        "customers": {
            "description": "Registered customers with demographic and geographic data",
            "columns": {
                "id": "Primary key, auto-increment integer",
                "name": "Full customer name",
                "email": "Unique email address",
                "region": "Geographic region: NA, EMEA, APAC, LATAM",
                "segment": "Customer segment: Enterprise, SMB, Consumer",
                "created_at": "Account registration date (YYYY-MM-DD)"
            },
            "keywords": ["customer", "buyer", "client", "account", "user", "region"]
        },
        "orders": {
            "description": "Purchase orders placed by customers",
            "columns": {
                "id": "Primary key",
                "customer_id": "FK → customers.id",
                "order_date": "Date order was placed (YYYY-MM-DD)",
                "status": "One of: pending, shipped, delivered, returned"
            },
            "keywords": ["order", "purchase", "buy", "transaction", "sale"]
        },
        "order_items": {
            "description": "Individual line items within an order",
            "columns": {
                "id": "Primary key",
                "order_id": "FK → orders.id",
                "product_id": "FK → products.id",
                "quantity": "Number of units",
                "unit_price": "Price per unit at time of purchase"
            },
            "keywords": ["item", "line item", "quantity", "price", "revenue", "amount"]
        },
        "products": {
            "description": "Product catalog",
            "columns": {
                "id": "Primary key",
                "name": "Product name",
                "category_id": "FK → categories.id",
                "base_price": "Standard retail price"
            },
            "keywords": ["product", "item", "goods", "merchandise", "sku"]
        },
        "categories": {
            "description": "Product category hierarchy",
            "columns": {
                "id": "Primary key",
                "name": "Category name (e.g., Electronics, Clothing)"
            },
            "keywords": ["category", "type", "group", "department", "classification"]
        }
    },
    "relationships": [
        "orders.customer_id → customers.id (each order belongs to one customer)",
        "order_items.order_id → orders.id (each item belongs to one order)",
        "order_items.product_id → products.id (each item references one product)",
        "products.category_id → categories.id (each product is in one category)"
    ],
    "glossary": {
        "revenue": "SUM(order_items.quantity * order_items.unit_price)",
        "AOV": "Average order value = revenue / COUNT(DISTINCT orders.id)",
        "active customer": "Has ≥1 order in the last 90 days",
        "churn": "No orders in the last 180 days",
        "repeat customer": "Has placed ≥2 orders lifetime"
    },
    "example_queries": [
        {"q": "Total revenue by region", "sql": "SELECT c.region, SUM(oi.quantity * oi.unit_price) AS revenue FROM customers c JOIN orders o ON c.id = o.customer_id JOIN order_items oi ON o.id = oi.order_id GROUP BY c.region;"},
        {"q": "Top 5 products by units sold", "sql": "SELECT p.name, SUM(oi.quantity) AS units FROM products p JOIN order_items oi ON p.id = oi.product_id GROUP BY p.name ORDER BY units DESC LIMIT 5;"},
        {"q": "Monthly order count", "sql": "SELECT strftime('%Y-%m', order_date) AS month, COUNT(*) AS orders FROM orders GROUP BY month ORDER BY month;"}
    ]
}

print(f"Schema store loaded:")
print(f"  Tables: {list(SCHEMA_STORE['tables'].keys())}")
print(f"  Relationships: {len(SCHEMA_STORE['relationships'])}")
print(f"  Glossary terms: {len(SCHEMA_STORE['glossary'])}")
print(f"  Example queries: {len(SCHEMA_STORE['example_queries'])}")

In [ ]:
def retrieve_schema_context(question: str, schema: dict) -> SchemaContext:
    """
    Retrieve relevant schema context for a question using keyword matching.

    In production, this would use embedding-based retrieval. Here we use
    keyword overlap for demonstration.

    Args:
        question: Natural-language question
        schema: Schema documentation store

    Returns:
        SchemaContext with relevant tables, columns, relationships, etc.
    """
    question_lower = question.lower()
    words = set(question_lower.split())

    # Score tables by keyword overlap
    table_scores: dict[str, int] = {}
    for table_name, table_info in schema["tables"].items():
        score = sum(1 for kw in table_info["keywords"] if kw in question_lower)
        # Also check column names
        score += sum(1 for col in table_info["columns"] if col in question_lower)
        if score > 0:
            table_scores[table_name] = score

    # Always include directly referenced tables + their join neighbors
    relevant_tables = sorted(table_scores, key=table_scores.get, reverse=True)
    if not relevant_tables:
        relevant_tables = list(schema["tables"].keys())  # fallback: all

    # Gather columns for relevant tables
    columns = {t: list(schema["tables"][t]["columns"].keys()) for t in relevant_tables}

    # Filter relationships to relevant tables
    relationships = [
        r for r in schema["relationships"]
        if any(t in r for t in relevant_tables)
    ]

    # Match glossary terms
    glossary = [
        f"{term}: {defn}"
        for term, defn in schema["glossary"].items()
        if term.lower() in question_lower or any(w in term.lower() for w in words)
    ]

    # Find similar example queries
    examples = [
        f"Q: {ex['q']}\nSQL: {ex['sql']}"
        for ex in schema["example_queries"]
        if any(w in ex["q"].lower() for w in words if len(w) > 3)
    ]

    return SchemaContext(
        tables=relevant_tables,
        columns=columns,
        relationships=relationships,
        glossary_terms=glossary,
        example_queries=examples
    )


# Test retrieval
test_questions = [
    "What is the total revenue by region?",
    "Show me the top products by sales",
    "How many customers signed up last month?",
    "What categories have the highest average order value?",
    "Which customers have churned?"
]

for q in test_questions:
    ctx = retrieve_schema_context(q, SCHEMA_STORE)
    print(f"Q: {q}")
    print(f"  Tables: {ctx.tables} | Glossary: {len(ctx.glossary_terms)} | Examples: {len(ctx.example_queries)}")

## 4 · Question disambiguation

Ambiguous questions lead to wrong SQL. The **disambiguator** detects vagueness
and makes explicit assumptions before SQL generation.

Strategy:
1. Detect ambiguous terms (e.g., "top", "recent", "best")
2. Generate explicit assumptions
3. Return a clarified question with assumptions stated

In [ ]:
AMBIGUOUS_TERMS: dict[str, list[str]] = {
    "top": ["by revenue", "by quantity", "by order count", "by rating"],
    "best": ["highest revenue", "highest rating", "most purchased"],
    "recent": ["last 30 days", "last 90 days", "last quarter"],
    "popular": ["most purchased", "most viewed", "highest rated"],
    "active": ["ordered in last 90 days", "logged in last 30 days"],
    "big": ["highest revenue", "most employees", "most orders"],
    "growth": ["revenue growth", "customer growth", "order growth"],
    "performance": ["revenue", "profit margin", "conversion rate"],
}


def disambiguate_question(question: str) -> DisambiguatedQuestion:
    """
    Detect ambiguous terms and create explicit assumptions.

    Args:
        question: Raw user question

    Returns:
        DisambiguatedQuestion with clarified text and listed assumptions
    """
    question_lower = question.lower()
    assumptions: list[str] = []
    clarified = question

    for term, options in AMBIGUOUS_TERMS.items():
        if term in question_lower:
            default = options[0]
            assumptions.append(f'"{term}" interpreted as: {default}')
            clarified = clarified  # Keep original; assumptions are separate

    # Add default time scope if no time mentioned
    time_words = {"today", "yesterday", "week", "month", "quarter", "year", "2024", "2023"}
    if not any(tw in question_lower for tw in time_words):
        assumptions.append("Time scope: all available data (no time filter specified)")

    # Add default limit if "top" or "best" without number
    if ("top" in question_lower or "best" in question_lower):
        import re
        if not re.search(r"\b\d+\b", question):
            assumptions.append("Limit: top 10 (no specific count mentioned)")

    return DisambiguatedQuestion(
        original=question,
        clarified=clarified,
        assumptions=assumptions
    )


# Test disambiguation
test_qs = [
    "Show me the top customers",
    "What are the best products this quarter?",
    "Which regions have the most growth?",
    "How many active customers do we have?",
    "Show total revenue for 2024"
]

for q in test_qs:
    result = disambiguate_question(q)
    print(f"Q: {result.original}")
    for a in result.assumptions:
        print(f"  ⚬ {a}")
    print()

## 5 · SQL generation with schema context

The SQL generator combines:
1. **Schema context** from the retriever
2. **Disambiguation** assumptions
3. **Few-shot examples** of similar queries
4. A carefully structured prompt

In [ ]:
def build_sql_prompt(
    question: str,
    schema_ctx: SchemaContext,
    disambiguation: DisambiguatedQuestion
) -> str:
    """
    Build the SQL generation prompt with full context.

    Args:
        question: Original user question
        schema_ctx: Retrieved schema context
        disambiguation: Disambiguation results with assumptions

    Returns:
        Formatted prompt string for the LLM
    """
    sections: list[str] = []

    # Schema section
    sections.append("=== DATABASE SCHEMA ===")
    for table in schema_ctx.tables:
        cols = ", ".join(schema_ctx.columns.get(table, []))
        sections.append(f"TABLE {table} ({cols})")

    # Relationships
    if schema_ctx.relationships:
        sections.append("\n=== RELATIONSHIPS ===")
        for rel in schema_ctx.relationships:
            sections.append(f"  {rel}")

    # Business glossary
    if schema_ctx.glossary_terms:
        sections.append("\n=== BUSINESS DEFINITIONS ===")
        for term in schema_ctx.glossary_terms:
            sections.append(f"  {term}")

    # Example queries
    if schema_ctx.example_queries:
        sections.append("\n=== EXAMPLE QUERIES ===")
        for ex in schema_ctx.example_queries:
            sections.append(ex)

    # Assumptions
    if disambiguation.assumptions:
        sections.append("\n=== ASSUMPTIONS ===")
        for a in disambiguation.assumptions:
            sections.append(f"  • {a}")

    # Instructions
    sections.append("\n=== INSTRUCTIONS ===")
    sections.append("Generate a single SQLite-compatible SQL query.")
    sections.append("Use only the tables and columns listed above.")
    sections.append("Return ONLY the SQL, no explanation.")

    sections.append(f"\n=== QUESTION ===\n{question}")

    return "\n".join(sections)


# Test prompt construction
test_q = "What are the top products by revenue?"
ctx = retrieve_schema_context(test_q, SCHEMA_STORE)
disamb = disambiguate_question(test_q)
prompt = build_sql_prompt(test_q, ctx, disamb)

print("Generated prompt:")
print("─" * 60)
print(prompt)
print("─" * 60)
print(f"\nPrompt length: {len(prompt)} characters")

In [ ]:
def generate_sql(
    question: str,
    schema_ctx: SchemaContext,
    disambiguation: DisambiguatedQuestion,
    error_context: Optional[str] = None
) -> str:
    """
    Generate SQL from a natural-language question using an LLM.

    Args:
        question: User's question
        schema_ctx: Retrieved schema context
        disambiguation: Disambiguation with assumptions
        error_context: Previous error for self-correction (optional)

    Returns:
        Generated SQL string
    """
    prompt = build_sql_prompt(question, schema_ctx, disambiguation)

    if error_context:
        prompt += f"\n\n=== PREVIOUS ERROR (fix this) ===\n{error_context}"

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You are a SQL expert. Generate only valid SQLite SQL. "
                "Return ONLY the SQL query, no markdown, no explanation."
            )},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=500
    )

    sql = response.choices[0].message.content.strip()
    # Clean up common LLM artifacts
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql


# Test on 5 questions
test_questions = [
    "How many customers do we have?",
    "What is the total revenue by region?",
    "Show me the top 5 products by revenue",
    "What is the monthly revenue trend?",
    "Which customers have placed the most orders?"
]

for q in test_questions:
    ctx = retrieve_schema_context(q, SCHEMA_STORE)
    disamb = disambiguate_question(q)
    sql = generate_sql(q, ctx, disamb)
    print(f"Q: {q}")
    print(f"  SQL: {sql[:100]}{'...' if len(sql) > 100 else ''}")
    print()

## 6 · Execution and validation

After generating SQL, we execute it and validate the results:

| Check | What it catches |
|---|---|
| **Syntax valid** | Malformed SQL |
| **Executes without error** | Wrong column/table names |
| **Non-empty result** | Wrong JOINs, over-filtering |
| **Reasonable row count** | Missing GROUP BY, cartesian products |
| **Column types match** | Aggregation errors |

In [ ]:
# Create a test database for validation demos

conn = sqlite3.connect(":memory:")
conn.executescript("""
    CREATE TABLE customers (
        id INTEGER PRIMARY KEY, name TEXT, email TEXT,
        region TEXT, segment TEXT, created_at DATE
    );
    CREATE TABLE orders (
        id INTEGER PRIMARY KEY, customer_id INTEGER,
        order_date DATE, status TEXT
    );
    CREATE TABLE order_items (
        id INTEGER PRIMARY KEY, order_id INTEGER,
        product_id INTEGER, quantity INTEGER, unit_price REAL
    );
    CREATE TABLE products (
        id INTEGER PRIMARY KEY, name TEXT,
        category_id INTEGER, base_price REAL
    );
    CREATE TABLE categories (id INTEGER PRIMARY KEY, name TEXT);

    INSERT INTO categories VALUES (1,'Electronics'),(2,'Clothing'),(3,'Books');
    INSERT INTO products VALUES (1,'Laptop',1,999.99),(2,'T-Shirt',2,29.99),
        (3,'Python Book',3,49.99),(4,'Phone',1,699.99),(5,'Jacket',2,89.99);
    INSERT INTO customers VALUES (1,'Alice','alice@ex.com','NA','Enterprise','2023-01-15'),
        (2,'Bob','bob@ex.com','EMEA','SMB','2023-03-20'),
        (3,'Carol','carol@ex.com','APAC','Consumer','2023-06-10');
    INSERT INTO orders VALUES (1,1,'2024-01-15','delivered'),(2,1,'2024-03-20','delivered'),
        (3,2,'2024-02-10','shipped'),(4,3,'2024-04-05','pending');
    INSERT INTO order_items VALUES (1,1,1,1,999.99),(2,1,3,2,49.99),
        (3,2,4,1,699.99),(4,3,2,3,29.99),(5,4,5,1,89.99);
""")

print("✅ Test database created")
print(f"  Tables: customers, orders, order_items, products, categories")
df = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"  Verified: {df['name'].tolist()}")

In [ ]:
@dataclass
class ValidationResult:
    """Result of SQL validation checks."""
    is_valid: bool
    checks_passed: list[str]
    checks_failed: list[str]
    data: Optional[pd.DataFrame] = None
    error: Optional[str] = None


def validate_sql_result(
    sql: str,
    conn: sqlite3.Connection,
    max_rows: int = 10_000
) -> ValidationResult:
    """
    Execute SQL and run validation checks on the result.

    Args:
        sql: SQL query to validate
        conn: SQLite connection
        max_rows: Maximum expected rows (catches cartesian products)

    Returns:
        ValidationResult with pass/fail details
    """
    passed: list[str] = []
    failed: list[str] = []

    # Check 1: Syntax — can it be parsed/executed?
    try:
        df = pd.read_sql_query(sql, conn)
        passed.append("syntax_valid")
    except Exception as e:
        failed.append(f"syntax_error: {e}")
        return ValidationResult(False, passed, failed, error=str(e))

    # Check 2: Non-empty result
    if df.empty:
        failed.append("empty_result: query returned 0 rows")
    else:
        passed.append(f"non_empty: {len(df)} rows")

    # Check 3: Reasonable row count
    if len(df) > max_rows:
        failed.append(f"too_many_rows: {len(df)} > {max_rows} (possible cartesian product)")
    else:
        passed.append(f"row_count_ok: {len(df)} ≤ {max_rows}")

    # Check 4: No all-NULL columns
    null_cols = [col for col in df.columns if df[col].isna().all()]
    if null_cols:
        failed.append(f"all_null_columns: {null_cols}")
    else:
        passed.append("no_all_null_columns")

    is_valid = len(failed) == 0
    return ValidationResult(is_valid, passed, failed, data=df)


# Test validation with good and bad queries
test_sqls = [
    ("Good query", "SELECT c.region, COUNT(*) AS orders FROM customers c JOIN orders o ON c.id = o.customer_id GROUP BY c.region;"),
    ("Bad column", "SELECT nonexistent FROM customers;"),
    ("Empty result", "SELECT * FROM customers WHERE region = 'MARS';"),
]

for label, sql in test_sqls:
    result = validate_sql_result(sql, conn)
    status = "✅" if result.is_valid else "❌"
    print(f"{status} {label}")
    print(f"  Passed: {result.checks_passed}")
    print(f"  Failed: {result.checks_failed}")
    print()

## 7 · Self-correction agent

The self-correction agent wraps generation + execution + validation in a retry loop.
On failure, it feeds the error back to the LLM for another attempt (max 3 tries).

```
┌─── Attempt 1 ──────────────────────────────────────────┐
│ Generate SQL → Execute → Validate                      │
│   ✅ Pass → return result                              │
│   ❌ Fail → extract error → feed back to LLM           │
├─── Attempt 2 ──────────────────────────────────────────┤
│ Regenerate with error context → Execute → Validate     │
│   ✅ Pass → return result                              │
│   ❌ Fail → extract error → feed back to LLM           │
├─── Attempt 3 (final) ─────────────────────────────────┤
│ Regenerate with cumulative errors → Execute → Validate │
│   Return whatever we have (success or failure)         │
└────────────────────────────────────────────────────────┘
```

In [ ]:
def self_correcting_sql_agent(
    question: str,
    conn: sqlite3.Connection,
    schema: dict,
    max_attempts: int = 3
) -> SQLResult:
    """
    Generate, execute, validate, and self-correct SQL queries.

    Args:
        question: Natural-language question
        conn: Database connection for execution
        schema: Schema documentation store
        max_attempts: Maximum generation attempts

    Returns:
        SQLResult with final SQL, data, and attempt count
    """
    schema_ctx = retrieve_schema_context(question, schema)
    disambiguation = disambiguate_question(question)
    error_context: Optional[str] = None

    for attempt in range(1, max_attempts + 1):
        print(f"  Attempt {attempt}/{max_attempts}...")

        # Generate
        sql = generate_sql(question, schema_ctx, disambiguation, error_context)
        print(f"    SQL: {sql[:80]}{'...' if len(sql) > 80 else ''}")

        # Execute + Validate
        validation = validate_sql_result(sql, conn)

        if validation.is_valid:
            print(f"    ✅ Valid — {len(validation.data)} rows returned")
            return SQLResult(
                sql=sql, success=True,
                data=validation.data, attempts=attempt
            )

        # Prepare error context for next attempt
        error_msg = "; ".join(validation.checks_failed)
        error_context = f"SQL: {sql}\nError: {error_msg}"
        print(f"    ❌ Failed: {error_msg}")

    return SQLResult(
        sql=sql, success=False,
        error=error_msg, attempts=max_attempts
    )


# Test the self-correcting agent
print("Testing self-correcting SQL agent:\n")
questions = [
    "How many customers do we have?",
    "What is the total revenue?",
    "Show me the average order value by region"
]

for q in questions:
    print(f"Q: {q}")
    result = self_correcting_sql_agent(q, conn, SCHEMA_STORE)
    print(f"  Result: {'✅ Success' if result.success else '❌ Failed'} in {result.attempts} attempt(s)")
    if result.data is not None:
        print(f"  Data:\n{result.data.to_string(index=False)}")
    print()

conn.close()

## 🏋️ Exercises

1. **Embedding-based retrieval**: Replace the keyword-matching `retrieve_schema_context`
   with an embedding-based retriever using `sentence-transformers`. Compare relevance
   on 10 test questions.

2. **LLM disambiguation**: Replace the rule-based `disambiguate_question` with an LLM call
   that detects ambiguity and generates assumptions. Compare quality on 10 ambiguous questions.

3. **Multi-database support**: Extend the architecture to support PostgreSQL and MySQL
   dialects. What changes are needed in the prompt and validator?

## 🎯 Takeaways

- A production text-to-SQL system needs six coordinated components
- Schema RAG retrieves only the relevant context, keeping prompts focused
- Disambiguation makes implicit assumptions explicit before generation
- Few-shot examples in the prompt dramatically improve SQL quality
- Validation catches errors before they reach the user
- Self-correction with error feedback fixes most generation failures in ≤3 attempts

## ⏭️ What's next

In **02_build.ipynb**, we'll build the complete AI data analyst: a realistic
e-commerce database, full schema RAG, the text-to-SQL agent, insight narration,
and auto-chart generation — all working end-to-end.

## 📚 References

1. Gao, D. et al. (2023). "Text-to-SQL Empowered by Large Language Models: A Benchmark Evaluation." arXiv:2308.15363
2. Li, J. et al. (2024). "Can LLM Already Serve as A Database Interface? A BIg Bench for Large-Scale Database Grounded Text-to-SQLs." NeurIPS.
3. Lewis, P. et al. (2020). "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks." NeurIPS.
4. Rajkumar, N. et al. (2022). "Evaluating the Text-to-SQL Capabilities of Large Language Models." arXiv:2204.00498